In [5]:
import torch

print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")


torch: 2.9.1+cu126
cuda available: True
gpu: NVIDIA GeForce RTX 3060


라이브러리 불러오기

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


csv 읽고 colum 확인

In [2]:
INPUT_CSV = "recipes_output_fixed.csv"
TEXT_COL = "doc_text"     # 임베딩할 텍스트
ID_COL = "RCP_SNO"        # 있으면 좋고, 없어도 동작하게 처리할 거야

df = pd.read_csv(INPUT_CSV, encoding="utf-8-sig")
print("rows, cols:", df.shape)
print("columns:", df.columns.tolist())
df.head(3)


rows, cols: (5000, 9)
columns: ['RCP_SNO', 'CKG_NM', 'ingredients_keywords', 'doc_text', 'kind', 'situation', 'method', 'main_material_group', 'servings']


임베딩용 데이터 전처리(결측 제거 + passage 접두어)

In [3]:
# doc_text 없는 행은 제외
df2 = df.dropna(subset=[TEXT_COL]).copy()

# id 컬럼이 없으면 임시 id 생성
if ID_COL not in df2.columns:
    df2[ID_COL] = np.arange(len(df2))

# E5 권장: 문서에 "passage: " 접두어
passages = ["passage: " + str(t) for t in df2[TEXT_COL].tolist()]

print("rows for embedding:", len(passages))
print("sample passage:\n", passages[0][:300])


rows for embedding: 5000
sample passage: (output removed - contains recipe data)


모델 load

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "intfloat/multilingual-e5-large"

print("device:", device)
model = SentenceTransformer(model_name, device=device)


device: cuda


임베딩 생성(GPU 배치 조절)

In [8]:
BATCH_SIZE = 64

embeddings = model.encode(
    passages,
    batch_size=BATCH_SIZE,
    show_progress_bar=True,
    normalize_embeddings=True
).astype(np.float32)

print("embeddings shape:", embeddings.shape)
print("dtype:", embeddings.dtype)


Batches:   0%|          | 0/79 [00:00<?, ?it/s]

embeddings shape: (5000, 1024)
dtype: float32


저장(벡터+메타)

In [9]:
OUT_EMB = "recipe_embeddings_e5_large.npy"
OUT_META = "recipe_meta.csv"

# 벡터 저장
np.save(OUT_EMB, embeddings)

# 메타 저장: doc_text는 크니까 제외(원하면 포함 가능)
meta_cols = [c for c in df2.columns if c != TEXT_COL]
df2[meta_cols].to_csv(OUT_META, index=False, encoding="utf-8-sig")

print("✅ saved:", OUT_EMB)
print("✅ saved:", OUT_META)


✅ saved: recipe_embeddings_e5_large.npy
✅ saved: recipe_meta.csv


저장 검증

In [10]:
x = np.load(OUT_EMB)
print("loaded embeddings:", x.shape, x.dtype)

meta = pd.read_csv(OUT_META, encoding="utf-8-sig")
print("loaded meta:", meta.shape)
meta.head(10)


loaded embeddings: (5000, 1024) float32
loaded meta: (5000, 8)


쿼리 임베딩 샘플

In [11]:
q = "query: 오늘은 매운 국물 음식 추천해줘 (spicy=4, soup=4, rich=1, context=3)"
q_vec = model.encode([q], normalize_embeddings=True).astype(np.float32)
q_vec.shape


(1, 1024)